### Process HRRRAK Grib Files: Juneau Icefield Domain

Notebook contents 
* Making a copy of other process script for the juneau icefield domain

created by Cassie Lumbrazo\
last updated: March 2026\
run location: UAS linux\
python environment: **rasterio**

In [1]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_theme()
# plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy

from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from matplotlib import ticker

import rioxarray
import rasterio 
import cfgrib
import os

In [2]:
pwd

'/home/cassie/python/repos/snow_modeling_point/icefield_domain'

### Open all the HRRR-AK Grib Files for a Water Year (WY) and Create a Simple NetCDF

To scale up the process for multiple HRRR .grib2 files across a folder structure, we need to:

* Walk through directories inside folderspath (`/hdd/snow_hydrology/hrrrak/icefield_domain/f567/WY2024`).
* Identify .grib2 files inside each date-named subfolder (`hrrr.20231001`, `hrrr.20231002`, etc.).
* Open and process all files for a given date just like in your original script.
* Merge all processed datasets and store them in a structured way.

* and deal with CRS issues


In [3]:
import xarray as xr
import numpy as np
import pygrib
import os
from concurrent.futures import ProcessPoolExecutor
import rioxarray  # For CRS handling
import pandas as pd
import functools

# Desired variables (short names from GRIB output, keeping naming conventions)
desired_vars = [
    't',        # Temperature (surface level)
    'sp',       # Surface pressure
    '2t',       # 2 metre temperature
    '2r',       # 2 metre relative humidity
    'tp',       # Total Precipitation
    'prate',    # Precipitation rate
    '10u',      # 10 metre U wind component
    '10v',      # 10 metre V wind component
    'sdswrf',   # Surface downward short-wave radiation flux
    'sdlwrf',   # Surface downward long-wave radiation flux
    # Extras
    'gust',     # Wind speed (gust)
    'tcc',      # Total Cloud Cover
    'lcc',      # Low cloud cover
    'mcc',      # Medium cloud cover
    'hcc',      # High cloud cover
    'lai',      # Leaf Area Index
    '2d',       # 2 metre dewpoint temperature
    '2sh',      # 2 metre specific humidity
    'suswrf',   # Surface upward short-wave radiation flux
    'sulwrf',   # Surface upward long-wave radiation flux
    'orog',     # Orography
    'sdwe',     # Water equivalent of accumulated snow depth
    'sde',      # Snow depth
    'veg',      # Vegetation
    'vgtyp',    # Vegetation Type
    'gflux',    # Ground heat flux
]


def process_grib_file(grib_path):
    """ Process a single .grib2 file and return an xarray dataset """
    try:
        # Extract init time and forecast step from filename
        # Example: 'hrrr.t00z.wrfsfcf05.ak.grib2' -> init_hour=0, forecast_step=5
        filename = os.path.basename(grib_path)
        init_hour = int(filename.split('.')[1][1:3])  # 't00z' -> 0
        forecast_step = int(filename.split('.')[2][-2:])  # 'wrfsfcf05' -> '05' -> 5

        # Get date from folder name
        date_str = grib_path.split('/')[-2].split('.')[1]  # e.g., '20241001'
        init_datetime = pd.to_datetime(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]} {init_hour:02d}:00")
        valid_time = init_datetime + pd.Timedelta(hours=forecast_step)

        # Collect desired vars using pygrib
        grbs = pygrib.open(grib_path)
        var_info = {}
        for grb in grbs:
            short_name = grb.shortName.lower()
            if short_name in desired_vars:
                # Special filter: for 't', only keep surface level
                if short_name == 't' and not (grb.typeOfLevel == 'surface'):
                    continue
                key = f"{short_name}_{grb.level}_{grb.typeOfLevel}"
                var_info[key] = {
                    'short_name': short_name,
                    'level': grb.level,
                    'typeOfLevel': grb.typeOfLevel,
                    'data': grb.values,
                    'lat': grb.latlons()[0],
                    'lon': grb.latlons()[1],
                    'attrs': {  # Preserve GRIB metadata
                        'long_name': getattr(grb, 'name', short_name),
                        'units': getattr(grb, 'units', ''),
                        'GRIB_shortName': getattr(grb, 'shortName', short_name),
                        'GRIB_name': getattr(grb, 'name', ''),
                        'GRIB_cfName': getattr(grb, 'cfName', ''),
                        'GRIB_cfVarName': getattr(grb, 'cfVarName', ''),
                        'level': grb.level,
                        'typeOfLevel': grb.typeOfLevel
                    }
                }
        grbs.close()

        if not var_info:
            print(f"⚠️ Skipping {grib_path}: No desired variables found")
            return None

        # Create merged dataset
        datasets = []
        first_key = list(var_info.keys())[0]
        lat = var_info[first_key]['lat']
        lon = var_info[first_key]['lon']

        grouped_vars = {}
        for key, info in var_info.items():
            group_key = f"{info['short_name']}_{info['typeOfLevel']}"
            if group_key not in grouped_vars:
                grouped_vars[group_key] = []
            grouped_vars[group_key].append(info)

        for group_key, infos in grouped_vars.items():
            short_name = infos[0]['short_name']
            type_level = infos[0]['typeOfLevel']
            attrs = infos[0]['attrs']  # Use attrs from first (assuming consistent)

            if len(infos) == 1:
                info = infos[0]
                data = info['data']
                level = info['level']

                if type_level == 'surface' or (type_level == 'heightAboveGround' and level == 0):
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                elif type_level == 'heightAboveGround' and level in [2, 10]:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs={**attrs, 'heightAboveGround': level},  # Add height as attr
                        name=short_name
                    )
                else:
                    da = xr.DataArray(
                        data,
                        dims=['y', 'x'],
                        coords={
                            'y': range(data.shape[0]),
                            'x': range(data.shape[1]),
                            'latitude': (['y', 'x'], lat),
                            'longitude': (['y', 'x'], lon)
                        },
                        attrs=attrs,
                        name=short_name
                    )
                datasets.append(da)
            else:
                # Multi-level (rare now)
                levels = [info['level'] for info in infos]
                data_stack = np.stack([info['data'] for info in infos], axis=0)
                da = xr.DataArray(
                    data_stack,
                    dims=['level', 'y', 'x'],
                    coords={
                        'level': levels,
                        'y': range(data_stack.shape[1]),
                        'x': range(data_stack.shape[2]),
                        'latitude': (['y', 'x'], lat),
                        'longitude': (['y', 'x'], lon)
                    },
                    attrs=attrs,
                    name=short_name
                )
                datasets.append(da)

        ds_merged = xr.merge(datasets, compat='override')

        # Calculate wind speed and direction (from 10u and 10v), with attrs
        if '10u' in ds_merged and '10v' in ds_merged:
            ds_merged['wind'] = np.sqrt(ds_merged['10u']**2 + ds_merged['10v']**2)
            ds_merged['wind'].attrs.update({
                'long_name': '10 metre wind speed calculated from u and v wind components',
                'units': 'm s**-1',
                'GRIB_shortName': '10m wind',
                'standard_name': 'wind speed',
                'GRIB_name': '10 metre wind speed',
                'GRIB_cfName': 'wind_speed',
                'GRIB_cfVarName': 'wind10'
            })

            # Wind direction: atan2(v, u) gives direction in degrees from east (positive x), convert to 0-360
            wind_dir_rad = np.arctan2(ds_merged['10v'], ds_merged['10u'])
            ds_merged['wind_dir'] = (wind_dir_rad * 180 / np.pi + 360) % 360
            ds_merged['wind_dir'].attrs.update({
                'long_name': '10 metre wind direction (from which wind blows)',
                'units': 'degrees',
                'GRIB_shortName': '10m wind dir',
                'standard_name': 'wind_from_direction',
                'GRIB_name': '10 metre wind direction',
                'GRIB_cfName': 'wind_from_direction',
                'GRIB_cfVarName': 'winddir10'
            })

        # Rename variables (as in old code) - attrs should be preserved
        ds_merged = ds_merged.rename({
            't': 'temp_surface',
            'sp': 'pressure',
            '2t': 'temp',
            '2r': 'rh',
            'sdwe': 'swe',
            'sde': 'snowdepth',
            'sdswrf': 'swrad',
            'sdlwrf': 'lwrad',
            'tp': 'precip_total',
            'prate': 'precip_rate'
        })

        # Drop unnecessary variables if any
        drop_vars = [var for var in ['unknown'] if var in ds_merged]
        ds_merged = ds_merged.drop_vars(drop_vars, errors='ignore')

        # Assign time coordinates (mimic old code: time as valid_time, add step)
        step = pd.Timedelta(hours=forecast_step)
        ds_merged = ds_merged.assign_coords({
            'time': valid_time,
            'step': step,
            'valid_time': valid_time
        })

        return ds_merged

    except Exception as e:
        print(f"⚠️ Error processing {grib_path}: {e}")
        return None


def process_date_folder(folderspath, date_folder):
    """ Process all grib files in a date folder and return a concatenated dataset """
    date_path = os.path.join(folderspath, date_folder)
    grib_files = sorted([os.path.join(date_path, f) for f in os.listdir(date_path) if f.endswith(".grib2")])

    with ProcessPoolExecutor() as executor:
        ds_list = list(filter(None, executor.map(process_grib_file, grib_files)))

    if ds_list:
        try:
            return xr.concat(ds_list, dim="time")
        except ValueError as e:
            print(f"⚠️ Skipping {date_folder} due to concatenation error: {e}")
            return None
    return None


def process_water_year(wy):
    """
    Process all HRRR-AK grib files for a given water year and save NetCDF outputs.

    Parameters
    ----------
    wy : int or str
        Water year (e.g. 2022, 2023, 2024, 2025).
        Expects data at: /hdd/snow_hydrology/hrrrak/icefield_domain/f567/WY{wy}

    Returns
    -------
    ds_utm : xr.Dataset
        Reprojected dataset (EPSG:32608).
    """
    folderspath = f'/hdd/snow_hydrology/hrrrak/icefield_domain/f567/WY{wy}'
    date_folders = sorted([f for f in os.listdir(folderspath) if f.startswith("hrrr.")])

    _process_date = functools.partial(process_date_folder, folderspath)

    with ProcessPoolExecutor() as executor:
        all_ds_list = list(filter(None, executor.map(_process_date, date_folders)))

    if not all_ds_list:
        print("⚠️ No valid datasets found.")
        return None

    final_ds = xr.concat(all_ds_list, dim="time")
    print("✅ Successfully merged all datasets!")

    # --- Save raw NetCDF before CRS processing ---
    raw_output_path = f"/hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY{wy}_raw.nc"
    final_ds.to_netcdf(raw_output_path)
    print(f"📁 Saved raw dataset to {raw_output_path}")

    # --- CRS Check and Reprojection ---
    print(f"Original CRS: {final_ds.rio.crs}")

    # Assign CRS if not set (assuming WGS84 lat/lon)
    final_ds.rio.write_crs("EPSG:4326", inplace=True)
    print("Assigned CRS: EPSG:4326")

    # Drop old integer x/y coords to avoid rename conflict
    final_ds = final_ds.drop_vars(['x', 'y'], errors='ignore')

    # Rename dimensions and set spatial dims
    final_ds = final_ds.rename({'longitude': 'x', 'latitude': 'y'})
    final_ds = final_ds.rio.set_spatial_dims(x_dim='x', y_dim='y')

    # Extract 1D x and y coordinates from the 2D grid
    x_1d = final_ds['x'].isel(y=0)  # same x values across rows
    y_1d = final_ds['y'].isel(x=0)  # same y values down columns

    # Assign 1D coords
    final_ds = final_ds.assign_coords(x=x_1d, y=y_1d)

    # Drop old 2D coords
    final_ds = final_ds.drop_vars(['x', 'y'])
    final_ds = final_ds.set_coords([])
    final_ds = final_ds.assign_coords({'x': x_1d, 'y': y_1d})

    # Reproject to UTM
    ds_utm = final_ds.rio.reproject("EPSG:32608")
    print("Reprojected to UTM (EPSG:32608)")

    # Save the reprojected dataset
    utm_output_path = f"/hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY{wy}_utm.nc"
    ds_utm.to_netcdf(utm_output_path)
    print(f"📁 Saved reprojected dataset to {utm_output_path}")

    return ds_utm


### Icefield Domain, f567, WY2019

In [4]:
ds_utm = process_water_year(2019)

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2019_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326


TypeError: only length-1 arrays can be converted to Python scalars

In [5]:
# open the raw netcdf and take a look 

ds = xr.open_dataset('/hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2019_raw.nc')
ds

<xarray.Dataset> Size: 4GB
Dimensions:       (time: 8736, y: 58, x: 32)
Coordinates:
  * time          (time) datetime64[ns] 70kB 2018-10-01T05:00:00 ... 2019-10-...
    step          (time) timedelta64[ns] 70kB ...
    valid_time    (time) datetime64[ns] 70kB ...
    latitude      (y, x, time) float64 130MB ...
    longitude     (y, x, time) float64 130MB ...
  * y             (y) int32 232B 0 1 2 3 4 5 6 7 8 ... 50 51 52 53 54 55 56 57
  * x             (x) int32 128B 0 1 2 3 4 5 6 7 8 ... 24 25 26 27 28 29 30 31
Data variables: (12/26)
    gust          (time, y, x) float64 130MB ...
    pressure      (time, y, x) float64 130MB ...
    orog          (time, y, x) float64 130MB ...
    temp_surface  (time, y, x) float64 130MB ...
    swe           (time, y, x) float64 130MB ...
    snowdepth     (time, y, x) float64 130MB ...
    ...            ...
    swrad         (time, y, x) float64 130MB ...
    lwrad         (time, y, x) float64 130MB ...
    suswrf        (time, y, x) float64 130MB ...
    sulwrf        (time, y, x) float64 130MB ...
    wind          (time, y, x) float64 130MB ...
    wind_dir      (time, y, x) float64 130MB ...
Attributes:
    long_name:       Wind speed (gust)
    units:           m s**-1
    GRIB_shortName:  gust
    GRIB_name:       Wind speed (gust)
    GRIB_cfName:     unknown
    GRIB_cfVarName:  gust
    level:           0
    typeOfLevel:     surface

In [8]:
# open the 2020 raw dataset and see if anything looks different 
ds_2020 = xr.open_dataset('/hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2021_raw.nc')
ds_2020

<xarray.Dataset> Size: 4GB
Dimensions:       (time: 8760, y: 58, x: 31)
Coordinates:
  * time          (time) datetime64[ns] 70kB 2020-10-01T05:00:00 ... 2021-10-...
    step          (time) timedelta64[ns] 70kB ...
    valid_time    (time) datetime64[ns] 70kB ...
    latitude      (y, x) float64 14kB ...
    longitude     (y, x) float64 14kB ...
  * y             (y) int32 232B 0 1 2 3 4 5 6 7 8 ... 50 51 52 53 54 55 56 57
  * x             (x) int32 124B 0 1 2 3 4 5 6 7 8 ... 23 24 25 26 27 28 29 30
Data variables: (12/28)
    gust          (time, y, x) float64 126MB ...
    pressure      (time, y, x) float64 126MB ...
    orog          (time, y, x) float64 126MB ...
    temp_surface  (time, y, x) float64 126MB ...
    swe           (time, y, x) float64 126MB ...
    snowdepth     (time, y, x) float64 126MB ...
    ...            ...
    suswrf        (time, y, x) float64 126MB ...
    sulwrf        (time, y, x) float64 126MB ...
    wind          (time, y, x) float64 126MB ...
    wind_dir      (time, y, x) float64 126MB ...
    veg           (time, y, x) float64 126MB ...
    lai           (time, y, x) float64 126MB ...
Attributes:
    long_name:       Wind speed (gust)
    units:           m s**-1
    GRIB_shortName:  gust
    GRIB_name:       Wind speed (gust)
    GRIB_cfName:     unknown
    GRIB_cfVarName:  gust
    level:           0
    typeOfLevel:     surface

### Okay, the data structure is a bit off for WY2019... so for now, let's process without it because we would need to update the entire processing workflow then merge things with different grids and sizes, etc.

____________________________________________________________________

### Icefield Domain, f567, WY2020

In [9]:
ds_utm = process_water_year(2020)

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2020_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2020_utm.nc


### Icefield Domain, f567, WY2021

In [8]:
ds_utm = process_water_year(2021)

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2021_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2021_utm.nc


### Icefield Domain, f567, WY2022

In [ ]:
ds_utm = process_water_year(2022)

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2022_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2022_utm.nc


### Icefield Domain, f567, WY2023

In [ ]:
ds_utm = process_water_year(2023)

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2023_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2023_utm.nc


### Icefield Domain, f567, WY2024

In [ ]:
ds_utm = process_water_year(2024)

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2024_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2024_utm.nc


In [5]:
# open the dataset and take a look 
ds_utm

<xarray.Dataset> Size: 4GB
Dimensions:       (x: 31, y: 59, time: 8783)
Coordinates:
  * x             (x) float64 248B 4.98e+05 5.009e+05 ... 5.842e+05 5.871e+05
  * y             (y) float64 472B 6.618e+06 6.615e+06 ... 6.449e+06 6.446e+06
  * time          (time) datetime64[ns] 70kB 2023-10-01T05:00:00 ... 2024-10-...
    step          (time) timedelta64[ns] 70kB 05:00:00 06:00:00 ... 07:00:00
    valid_time    (time) datetime64[ns] 70kB 2023-10-01T05:00:00 ... 2024-10-...
    spatial_ref   int64 8B 0
Data variables: (12/28)
    gust          (time, y, x) float64 129MB 4.589 3.839 3.026 ... nan nan nan
    pressure      (time, y, x) float64 129MB 9.057e+04 8.889e+04 ... nan nan
    orog          (time, y, x) float64 129MB 912.1 1.063e+03 ... nan nan
    temp_surface  (time, y, x) float64 129MB 274.8 274.1 274.4 ... nan nan nan
    swe           (time, y, x) float64 129MB 0.0002 0.0104 0.0024 ... nan nan
    snowdepth     (time, y, x) float64 129MB 0.0 0.0 0.0 0.0 ... nan nan nan nan
    ...            ...
    swrad         (time, y, x) float64 129MB 0.0 0.0 0.0 0.0 ... nan nan nan nan
    lwrad         (time, y, x) float64 129MB 303.7 303.5 304.0 ... nan nan nan
    suswrf        (time, y, x) float64 129MB 0.0 0.0 0.0 0.0 ... nan nan nan nan
    sulwrf        (time, y, x) float64 129MB 307.1 303.9 305.3 ... nan nan nan
    wind          (time, y, x) float64 129MB 1.999 1.988 1.921 ... nan nan nan
    wind_dir      (time, y, x) float64 129MB 10.54 8.774 26.79 ... nan nan nan
Attributes:
    long_name:       Wind speed (gust)
    units:           m s**-1
    GRIB_shortName:  gust
    GRIB_name:       Wind speed (gust)
    GRIB_cfName:     unknown
    GRIB_cfVarName:  gust
    level:           0
    typeOfLevel:     surface

### Icefield Domain, f567, WY2025

In [ ]:
ds_utm = process_water_year(2025)

✅ Successfully merged all datasets!
📁 Saved raw dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2025_raw.nc
Original CRS: None
Assigned CRS: EPSG:4326
Reprojected to UTM (EPSG:32608)
📁 Saved reprojected dataset to /hdd/snow_hydrology/hrrrak/icefield_domain/netcdf/hrrrak_merged_grib_f567_WY2025_utm.nc
